In [66]:
import sys
import os

import pandas as pd

sys.path.append(os.path.abspath(os.path.join('..')))

sys.modules.pop("functionality.models", None)
sys.modules.pop("functionality.data_preparation", None)
from functionality.data_preparation import Emb_Dataset, train_model
from functionality.models import ChemBertaBinaryClassifierLightning

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

In [37]:
binds_0 = pd.read_parquet("../intermediates/downsampled_0_50_mln")
binds_1 = pd.read_parquet("../intermediates/1_class")
final_data = pd.concat([binds_0, binds_1], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
del binds_1
del binds_0

In [ ]:
chemberta_model = ChemBertaBinaryClassifierLightning()
protein_names = final_data.protein_name.unique()

trained_models = []

models = {
    "ChemBERTa": chemberta_model
}

for protein_name in protein_names:
    print(f"Training models for protein: {protein_name}")
    
    # Filter protein-specific data
    protein_data = final_data[final_data.protein_name == protein_name]

    train_data, val_data = train_test_split(protein_data, test_size=0.1, random_state=42)
    model_name = "seyonec/PubChem10M_SMILES_BPE_450k"
    train_dataset = Emb_Dataset(train_data, model_name)
    val_dataset = Emb_Dataset(val_data, model_name)

    train_loader = DataLoader(train_dataset, batch_size=100000, shuffle=True, drop_last = True)
    val_loader = DataLoader(val_dataset, batch_size=100000, shuffle=False, drop_last = True)
    
    # Train each model
    for model_name, model in models.items():
        print(f"Training model: {model_name} for protein {protein_name}")
        
        if model_name != 'lightgbm':
    
            trainer = train_model(model_name, protein_name, model, train_loader, val_loader)
            trained_models.append({
                "model_name": model_name,
                "protein_name": protein_name,
                "model": model
            })
        
            
    print(f"Completed training for protein: {protein_name}")

    models_dir = os.makedirs('../models', exist_ok=True)
    trainer.save_checkpoint(f'../models/{protein_name}_{model_name}.pkl')

print("Training complete")